To answer the 1st and 3rd sub-question The results of the annotations will be evaluated.

To run this notebook, the following files need to be imported:


**For the inter annotator agreement:**
*   first_200_before.json
*   first_200_after.csv


**For the LLM suggestions VS the gold standard:**
*   all_500_with_gpt.json


**For the evaluation:**
*   first_200_before.json
*   first_200_after.csv
*   golden_standard.csv


First, run the first cel with all the imports. The run the Evaluation and all the cells below. After these runs, the rest of the cell can be ran.






In [ ]:
import pandas as pd
from nltk.metrics import agreement
from nltk.metrics.distance import jaccard_distance, masi_distance
from statsmodels.stats.inter_rater import fleiss_kappa
from itertools import combinations
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import json
import statistics as st
from pathlib import Path

# Inter-Annotator Agreement


## Calculating Krippendorfs
MASI and Jaccard

(Pre-adjudication)

In [ ]:
# 1. Laad het JSON-bestand
with open('first_200_before.json', 'r') as f:
    data = json.load(f)

processed_rows = []

# 2. Loop door de data en extraheer de benodigde velden
for task in data:
    task_id = task.get('id')
    task_data = task.get('data', {})

    # Haal metadata op
    thread_title = task_data.get('thread_title')
    thread_body = task_data.get('thread_body')
    comment_body = task_data.get('comment_body')
    comment_id = task_data.get('comment_id')

    annotations = task.get('annotations', [])
    for ann in annotations:
        # Haal annotatie-details op
        ann_id = ann.get('id')
        annotator = ann.get('completed_by')
        created_at = ann.get('created_at')
        updated_at = ann.get('updated_at')
        lead_time = ann.get('lead_time')

        # Zoek specifiek naar 'comment_frames' in de resultaten
        comment_frames = None
        for res in ann.get('result', []):
            if res.get('from_name') == 'comment_frames':
                # Sla de waarde op als tekst (JSON-formaat)
                comment_frames = json.dumps(res.get('value', {}))

        # Voeg rij toe
        processed_rows.append({
            'annotation_id': ann_id,
            'annotator': annotator,
            'comment_body': comment_body,
            'comment_frames': comment_frames,
            'comment_id': comment_id,
            'created_at': created_at,
            'id': task_id,
            'lead_time': lead_time,
            'thread_body': thread_body,
            'thread_title': thread_title,
            'updated_at': updated_at
        })

# 3. Maak het DataFrame en exporteer naar CSV
df = pd.DataFrame(processed_rows)
df.to_csv('geconverteerd_bestand.csv', index=False)

print("Conversie voltooid! Bestand opgeslagen als 'geconverteerd_bestand.csv'")

Conversie voltooid! Bestand opgeslagen als 'geconverteerd_bestand.csv'


In [ ]:
df['clean_labels'] = df['comment_frames'].apply(clean_frames)

In [ ]:
unique_comments = df['comment_id'].unique()

To select the dataset for this analysis, uncomment only one of the lines below in the code cell:

*   For all 200 comments: Every line should have # at the beginning
*   For the first 150 comments: Remove the # from the first line.
*   For the last 50 comments: Remove the # from the second line.



In [ ]:
# first 150 comments
# df = df[df['comment_id'].isin(unique_comments[:150])]

# last 50 comments
# df = df[df['comment_id'].isin(unique_comments[150:200])]

In [ ]:
# pivot to table
pivot_df = df.pivot(index='comment_id', columns='annotator', values='clean_labels')

In [ ]:
# make datastructure into structure so that NLTK understands
task_data = []
for index, row in df.iterrows():
  labels = frozenset(row['clean_labels'])
  task_data.append((row['annotator'], row['comment_id'], labels))

In [ ]:
# initialise the task with MASI
task_masi = agreement.AnnotationTask(
    data=task_data,
    distance=masi_distance)

alpha_masi = task_masi.alpha()
print(f"Krippendorff's Alpha (MASI): {alpha_masi:.4f}")

Krippendorff's Alpha (MASI): 0.4356


In [ ]:
# initialise the task with Jaccard
task_jaccard = agreement.AnnotationTask(
    data=task_data,
    distance=jaccard_distance)

alpha_jaccard = task_jaccard.alpha()
print(f"Krippendorff's Alpha (Jaccard): {alpha_jaccard:.4f}")

Krippendorff's Alpha (Jaccard): 0.5093


## Calculating Fleiss Kappa

### Caclulation Inbetween Annotators

In [ ]:
# define function for Jaccard-similarity between two sets
def jaccard_set(s1, s2):
  s1, s2 = set(s1), set(s2)
  if not s1 and not s2: return 1.0
  return len(s1 & s2) / len(s1 | s2)

In [ ]:
# get annotator name
annotators = pivot_df.columns

In [ ]:
# calculate for each unique pair the mean Jaccard
pairwise_results = []
for p1, p2 in combinations(annotators, 2):
  scores = [jaccard_set(pivot_df.loc[cid, p1], pivot_df.loc[cid, p2]) for cid in pivot_df.index]

  mean_jaccard = np.mean(scores)
  pairwise_results.append({
      'pair': f'{p1}-{p2}',
      'mean_jaccard': mean_jaccard
  })

In [ ]:
pairwise_df = pd.DataFrame(pairwise_results)
display(pairwise_df.round(4))

,pair,mean_jaccard
0,1-2,0.6180
1,1-4,0.5754
2,2-4,0.5059


In [ ]:
# Calculate mean of pairwise Jaccard-scores
mean_jaccard = pairwise_df['mean_jaccard'].mean()
print(f"Mean Jaccard: {mean_jaccard:.4f}")

Mean Jaccard: 0.5664


### Caculating per Frame

In [ ]:
# get all frames
all_frames = set()
for sublist in df['clean_labels']:
  for frame in sublist:
    all_frames.add(frame)

In [ ]:
# initialise Binarizer
mlb = MultiLabelBinarizer()
mlb.fit([list(all_frames)])

MultiLabelBinarizer()

In [ ]:
kappa_results = []

pivot_df = df.pivot(index='comment_id', columns='annotator', values='clean_labels')

for frame in mlb.classes_:
  matrix = []
  for _, row in pivot_df.iterrows():
    votes = [1 if (isinstance(row[a], list) and frame in row[a]) else 0 for a in pivot_df.columns]
    n_yes = sum(votes)
    n_no = len(votes) - n_yes
    matrix.append([n_yes, n_no])

  kappa = fleiss_kappa(matrix)
  kappa_results.append({
      'Frame': frame,
      'Fleiss_Kappa': kappa
  })

In [ ]:
kappa_df = pd.DataFrame(kappa_results).sort_values('Fleiss_Kappa', ascending=False)
display(kappa_df.round(4))

,Frame,Fleiss_Kappa
3,Economic,0.7859
11,Political,0.7216
1,Crime and Punishment,0.6897
8,Morality,0.6238
7,Legality / Jurisprudence,0.6064
6,Health and Safety,0.5985
5,Fairness and Equality,0.5597
14,Security and Defense,0.5479
2,Cultural Identity,0.5387
9,Other,0.5211


## Calculating Krippendorfs
MASI and Jaccard

(Post-adjudication)

In [ ]:
# load the data
df = pd.read_csv('first_200_after.csv')

In [ ]:
df['clean_labels'] = df['comment_frames'].apply(clean_frames)

To select the dataset for this analysis, uncomment only one of the lines below in the code cell:

*   For all 200 comments: Every line should have # at the beginning
*   For the first 150 comments: Remove the # from the first line.
*   For the last 50 comments: Remove the # from the second line.



In [ ]:
# first 150 comments
# df = df[df['comment_id'].isin(unique_comments[:150])]

# last 50 comments
# df = df[df['comment_id'].isin(unique_comments[150:200])]

In [ ]:
# pivot to table
pivot_df = df.pivot(index='comment_id', columns='annotator', values='clean_labels')

In [ ]:
# make datastructure into structure so that NLTK understands
task_data = []
for index, row in df.iterrows():
  labels = frozenset(row['clean_labels'])
  task_data.append((row['annotator'], row['comment_id'], labels))

In [ ]:
# initialise the task with MASI
task_masi = agreement.AnnotationTask(
    data=task_data,
    distance=masi_distance)

alpha_masi = task_masi.alpha()
print(f"Krippendorff's Alpha (MASI): {alpha_masi:.4f}")

Krippendorff's Alpha (MASI): 0.4575


In [ ]:
# initialise the task with Jaccard
task_jaccard = agreement.AnnotationTask(
    data=task_data,
    distance=jaccard_distance)

alpha_jaccard = task_jaccard.alpha()
print(f"Krippendorff's Alpha (Jaccard): {alpha_jaccard:.4f}")

Krippendorff's Alpha (Jaccard): 0.5318


## Calculating Fleiss Kappa

### Cacluclation Inbetween Annotators

In [ ]:
# define function for Jaccard-similarity between two sets
def jaccard_set(s1, s2):
  s1, s2 = set(s1), set(s2)
  if not s1 and not s2: return 1.0
  return len(s1 & s2) / len(s1 | s2)

In [ ]:
# get annotator name
annotators = pivot_df.columns

In [ ]:
# calculate for each unique pair the mean Jaccard
pairwise_results = []
for p1, p2 in combinations(annotators, 2):
  scores = [jaccard_set(pivot_df.loc[cid, p1], pivot_df.loc[cid, p2]) for cid in pivot_df.index]

  mean_jaccard = np.mean(scores)
  pairwise_results.append({
      'pair': f'{p1}-{p2}',
      'mean_jaccard': mean_jaccard
  })

In [ ]:
pairwise_df = pd.DataFrame(pairwise_results)
display(pairwise_df.round(4))

,pair,mean_jaccard
0,1-2,0.6380
1,1-4,0.5954
2,2-4,0.5259


In [ ]:
# Calculate mean of pairwise Jaccard-scores
mean_jaccard = pairwise_df['mean_jaccard'].mean()
print(f"Mean Jaccard: {mean_jaccard:.4f}")

Mean Jaccard: 0.5864


### Caculating per Frame

In [ ]:
# get all frames
all_frames = set()
for sublist in df['clean_labels']:
  for frame in sublist:
    all_frames.add(frame)

In [ ]:
# initialise Binarizer
mlb = MultiLabelBinarizer()
mlb.fit([list(all_frames)])

MultiLabelBinarizer()

In [ ]:
kappa_results = []

pivot_df = df.pivot(index='comment_id', columns='annotator', values='clean_labels')

for frame in mlb.classes_:
  matrix = []
  for _, row in pivot_df.iterrows():
    votes = [1 if (isinstance(row[a], list) and frame in row[a]) else 0 for a in pivot_df.columns]
    n_yes = sum(votes)
    n_no = len(votes) - n_yes
    matrix.append([n_yes, n_no])

  kappa = fleiss_kappa(matrix)
  kappa_results.append({
      'Frame': frame,
      'Fleiss_Kappa': kappa
  })

In [ ]:
kappa_df = pd.DataFrame(kappa_results).sort_values('Fleiss_Kappa', ascending=False)
display(kappa_df.round(4))

,Frame,Fleiss_Kappa
3,Economic,0.7859
11,Political,0.7444
1,Crime and Punishment,0.6897
8,Morality,0.6349
7,Legality / Jurisprudence,0.6064
6,Health and Safety,0.5985
5,Fairness and Equality,0.5833
9,Other,0.5556
2,Cultural Identity,0.5519
14,Security and Defense,0.5479


# LLM suggestions VS gold standard

In [ ]:
# load json files
with open('all_500_with_gpt.json', 'r') as f:
  gpt_data = json.load(f)


In [ ]:
def find_choices(obj):
    # Recursively collect every value under any 'choices' key
    out = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == 'choices' and isinstance(v, list):
                out.update(x for x in v if isinstance(x, str))
            else:
                out |= find_choices(v)
    elif isinstance(obj, list):
        for x in obj:
            out |= find_choices(x)
    return out

def gold_of(record):
    # Human gold frames: the ground_truth annotation if present, else the union of all annotators
    anns = record.get('annotations', [])
    gt = [a for a in anns if a.get('ground_truth')]
    source = gt if gt else anns
    frames = set()
    for a in source:
        frames |= find_choices(a.get('result', []))
    return frames

def llm_of(record):
    # LLM-suggested frames from the gpt_predictions field
    return find_choices(record.get('gpt_predictions', []))

def jaccard(a, b):
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

In [ ]:
# Per-comment Jaccard for all records, then segment means
jac = [jaccard(gold_of(r), llm_of(r)) for r in gpt_data]

first200 = jac[:200]
next300   = jac[200:500]

print('Mean Jaccard (gold vs LLM)')
print(f'  First 200 (inner_id 1-200)   : {st.mean(first200):.4f}')
print(f'  201-500   (inner_id 201-500) : {st.mean(next300):.4f}')
print(f'  All 500                       : {st.mean(jac):.4f}')

Mean Jaccard (gold vs LLM)
  First 200 (inner_id 1-200)   : 0.5797
  201-500   (inner_id 201-500) : 0.5226
  All 500                       : 0.5455


In [ ]:
# Breakdown in blocks of 50 across the first 200 comments
for start in range(0, 500, 50):
    block = jac[start:start + 50]
    print(f'comments {start + 1:>3}-{start + 50:<3}: mean Jaccard {st.mean(block):.4f}')

comments   1-50 : mean Jaccard 0.6476
comments  51-100: mean Jaccard 0.6427
comments 101-150: mean Jaccard 0.5596
comments 151-200: mean Jaccard 0.4689
comments 201-250: mean Jaccard 0.6337
comments 251-300: mean Jaccard 0.6620
comments 301-350: mean Jaccard 0.4563
comments 351-400: mean Jaccard 0.3894
comments 401-450: mean Jaccard 0.4893
comments 451-500: mean Jaccard 0.5050


# Evaluation

In [ ]:
def clean_frames(frame_data):
  try:
    # try as JSON
    data = json.loads(frame_data)
    choices = data.get('choices', [])
    return sorted([str(s).strip() for s in choices])
  except:
    # if not JSON, try using it as plain text
    frames = [f.strip() for f in str(frame_data).split(',')]
    return sorted(f for f in frames if f)


Comment out the next cell if you want to calculate the scores for the data set pre-adjunction.

In [ ]:
# # load json
# with open('first_200_before.json', 'r') as f:
#     data = json.load(f)

# processed_rows = []

# # Loop through the data and make a list of the annotations
# for task in data:
#     task_data = task.get('data', {})
#     annotations = task.get('annotations', [])

#     for ann in annotations:
#         # Extract the results
#         comment_frames = None
#         if ann.get('result'):
#             comment_frames = json.dumps(ann['result'][0].get('value', {}))

#         processed_rows.append({
#             'annotation_id': ann.get('id'),
#             'annotator': ann.get('completed_by'),
#             'comment_body': task_data.get('comment_body'),
#             'comment_frames': comment_frames,
#             'comment_id': task_data.get('comment_id'),
#             'created_at': ann.get('created_at'),
#             'id': task.get('id'), # Dit is het task-id
#             'lead_time': ann.get('lead_time'),
#             'thread_body': task_data.get('thread_body'),
#             'thread_title': task_data.get('thread_title'),
#             'updated_at': ann.get('updated_at')
#         })

# # create dataframe
# annotators_df = pd.DataFrame(processed_rows)

Comment out the next cell if you want to calculate the scores for the data set post-adjunction

In [ ]:
# load the annotations of the first 200 comments
annotators_df = pd.read_csv('first_200_after.csv')

In [ ]:
# load the gold standard
gold_df = pd.read_csv('golden_standard.csv')

In [ ]:
# use def clean_frames on both files
annotators_df['clean_comment_frames'] = annotators_df['comment_frames'].apply(clean_frames)
gold_df['clean_gold_frames'] = gold_df['gold_frames'].apply(clean_frames)

In [ ]:
# merge the two tables together, based on comment_id
merged_df = pd.merge(
    annotators_df[['comment_id', 'annotator', 'comment_frames']],
    gold_df[['comment_id', 'gold_frames']],
    on='comment_id',
    how='inner'
)

In [ ]:
# check how the merge data looks like
# should have 600, because 3 annotators times 200 comments = 600 comments
print(f"total number of comments: {len(merged_df)}")

total number of comments: 600


In [ ]:
display(merged_df.head(6))

,comment_id,annotator,comment_frames,gold_frames
0,cuw8uls,2,"{""choices"":[""Cultural Identity"",""Public Opinio...","Cultural Identity, Political, Public Opinion"
1,cuw8uls,1,"{""choices"":[""Cultural Identity"",""Public Opinio...","Cultural Identity, Political, Public Opinion"
2,cuw8uls,4,"{""choices"":[""Cultural Identity"",""Political"",""P...","Cultural Identity, Political, Public Opinion"
3,d3vqnlv,2,"{""choices"":[""Economic"",""Capacity and Resources""]}","Capacity and Resources, Economic"
4,d3vqnlv,1,"{""choices"":[""Economic"",""Policy Prescription an...","Capacity and Resources, Economic"
5,d3vqnlv,4,"{""choices"":[""Economic"",""Capacity and Resources""]}","Capacity and Resources, Economic"


In [ ]:
# initialize binarizer
mlb = MultiLabelBinarizer()

# Apply clean_frames to the merged_df columns to get list format
merged_df['clean_gold_frames'] = merged_df['gold_frames'].apply(clean_frames)
merged_df['clean_comment_frames'] = merged_df['comment_frames'].apply(clean_frames)

all_labels = set()
# Use the newly created clean columns for collecting all labels
for l in merged_df['clean_gold_frames'].tolist() + merged_df['clean_comment_frames'].tolist():
  all_labels.update(l)
mlb.fit([list(all_labels)])

MultiLabelBinarizer()

In [ ]:
# make empty list to save results
results_list = []

In [ ]:
# dictionary to safe dataframe per annotator
annotator_dfs = {}

# group data per annotator
for annotator_id, group in merged_df.groupby('annotator'):

  # the ground truth
  y_true_bin = mlb.transform(group['clean_gold_frames'])

  # the annoations per annotator
  y_pred_bin = mlb.transform(group['clean_comment_frames'])

  # calculate total scores
  total_accuracy = accuracy_score(y_true_bin, y_pred_bin)
  total_precision = precision_score(y_true_bin, y_pred_bin, average='samples', zero_division=0)
  total_recall = recall_score(y_true_bin, y_pred_bin, average='samples', zero_division=0)
  total_f1 = f1_score(y_true_bin, y_pred_bin, average='samples', zero_division=0)

  # calculate scores per frame
  frame_precision = precision_score(y_true_bin, y_pred_bin, average=None, zero_division=0)
  frame_recall = recall_score(y_true_bin, y_pred_bin, average=None, zero_division=0)
  frame_f1 = f1_score(y_true_bin, y_pred_bin, average=None, zero_division=0)

  # temporary list for results for specific annotator
  annotator_results = []

  # loop through all frames
  for i, frame in enumerate(mlb.classes_):
    frame_accuracy = (y_true_bin[:, i] == y_pred_bin[:, i]).mean()

    annotator_results.append({
        'Frame': frame,
        'Accuracy': round(frame_accuracy, 4),
        'Precision': round(frame_precision[i], 4),
        'Recall': round(frame_recall[i], 4),
        'F1': round(frame_f1[i], 4)
    })

  annotator_results.append({
      'Frame': 'Total for all frames',
      'Accuracy': round(total_accuracy, 4),
      'Precision': round(total_precision, 4),
      'Recall': round(total_recall, 4),
      'F1': round(total_f1, 4)
  })

  # save results per annotator
  annotator_dfs[annotator_id] = pd.DataFrame(annotator_results)

In [ ]:
for annotator_id, df in annotator_dfs.items():
  print(f"Annotator: {annotator_id}")
  display(df)


Annotator: 1


,Frame,Accuracy,Precision,Recall,F1
0,Capacity and Resources,0.975,0.9000,0.6923,0.7826
1,Crime and Punishment,0.985,0.7500,0.8571,0.8000
2,Cultural Identity,0.940,0.9556,0.8113,0.8776
3,Economic,0.980,0.9286,0.9286,0.9286
4,External Regulation and Reputation,0.985,0.5000,0.6667,0.5714
5,Fairness and Equality,0.960,0.8333,0.9722,0.8974
6,Health and Safety,0.980,0.8947,0.8947,0.8947
7,Legality / Jurisprudence,0.950,0.9091,0.7143,0.8000
8,Morality,0.950,0.8571,0.9818,0.9153
9,Other,0.970,0.7917,0.9500,0.8636


Annotator: 2


,Frame,Accuracy,Precision,Recall,F1
0,Capacity and Resources,0.965,0.6667,0.9231,0.7742
1,Crime and Punishment,0.985,0.7000,1.0000,0.8235
2,Cultural Identity,0.890,0.7541,0.8679,0.8070
3,Economic,0.980,0.8750,1.0000,0.9333
4,External Regulation and Reputation,0.985,0.5000,0.6667,0.5714
5,Fairness and Equality,0.940,0.8529,0.8056,0.8286
6,Health and Safety,0.940,0.6522,0.7895,0.7143
7,Legality / Jurisprudence,0.965,0.8387,0.9286,0.8814
8,Morality,0.930,0.8596,0.8909,0.8750
9,Other,0.975,1.0000,0.7500,0.8571


Annotator: 4


,Frame,Accuracy,Precision,Recall,F1
0,Capacity and Resources,0.955,0.6429,0.6923,0.6667
1,Crime and Punishment,0.990,0.7778,1.0000,0.8750
2,Cultural Identity,0.900,0.7705,0.8868,0.8246
3,Economic,0.965,1.0000,0.7500,0.8571
4,External Regulation and Reputation,0.985,0.5000,1.0000,0.6667
5,Fairness and Equality,0.900,0.6818,0.8333,0.7500
6,Health and Safety,0.970,0.8421,0.8421,0.8421
7,Legality / Jurisprudence,0.950,0.8462,0.7857,0.8148
8,Morality,0.900,0.8723,0.7455,0.8039
9,Other,0.935,0.6667,0.7000,0.6829


# sub-RQ 3
the mean scores of the annotators


In [ ]:
all_data = pd.concat(annotator_dfs.values())

In [ ]:
mean_scores_df = all_data[all_data['Frame'] != 'Total for all frames'].groupby('Frame').mean().reset_index()

In [ ]:
total_mean_row = {
    'Frame': 'TOTAL (Mean of Annotators)',
    'Accuracy': round(all_data[all_data['Frame'] == 'Total for all frames']['Accuracy'].mean(), 4),
    'Precision': round(all_data[all_data['Frame'] == 'Total for all frames']['Precision'].mean(), 4),
    'Recall': round(all_data[all_data['Frame'] == 'Total for all frames']['Recall'].mean(), 4),
    'F1': round(all_data[all_data['Frame'] == 'Total for all frames']['F1'].mean(), 4)
}

In [ ]:
mean_scores_df = pd.concat([mean_scores_df, pd.DataFrame([total_mean_row])], ignore_index=True)

In [ ]:
print("\n--- Gemiddelde Scores van Alle Annotators Samen ---")
display(mean_scores_df.round(4))


--- Gemiddelde Scores van Alle Annotators Samen ---


,Frame,Accuracy,Precision,Recall,F1
0,Capacity and Resources,0.9650,0.7365,0.7692,0.7412
1,Crime and Punishment,0.9867,0.7426,0.9524,0.8328
2,Cultural Identity,0.9100,0.8267,0.8553,0.8364
3,Economic,0.9750,0.9345,0.8929,0.9063
4,External Regulation and Reputation,0.9850,0.5000,0.7778,0.6032
5,Fairness and Equality,0.9333,0.7893,0.8704,0.8253
6,Health and Safety,0.9633,0.7963,0.8421,0.8170
7,Legality / Jurisprudence,0.9550,0.8647,0.8095,0.8321
8,Morality,0.9267,0.8630,0.8727,0.8647
9,Other,0.9600,0.8195,0.8000,0.8012
